# Build enriched dataset
This notebook does two things:
1. Computes **genre popularity** per market per year from the Charts + Artists data
2. Builds the **enriched model dataset** by joining genre networks + distances + genre popularity

## 0. Config & imports

In [ ]:
import os
import ast
import glob
import pickle
import numpy as np
import pandas as pd
import torch
from lorentz import Lorentz

# ── paths — adjust MGD_ROOT to wherever you extracted the zip ────────────────
MGD_ROOT   = 'code/dataset/MGD'           # root of extracted MGD zip
CHARTS_DIR = os.path.join(MGD_ROOT, 'Charts')
ARTISTS_CSV = os.path.join(MGD_ROOT, 'Artists', 'spotify_artists_info_complete_reduced_genres.csv')
GENRE_NET_DIR = os.path.join(MGD_ROOT, 'Genre Collaboration Network', 'Original')

# embeddings (your existing files)
CKPT  = 'ckpt/final_enoa_graph.ckpt'
VOCAB = 'enoa_vocab.pkl'
DIM   = 2

MARKETS = ['au', 'br', 'ca', 'de', 'fr', 'gb', 'global', 'jp', 'us']
YEARS   = [2017, 2018, 2019]

# output
OUTPUT_POPULARITY = 'genre_popularity.csv'
OUTPUT_DATASET    = 'model_dataset.csv'
# ─────────────────────────────────────────────────────────────────────────────

print('Config OK')

## 1. Load embeddings

In [ ]:
genres_list = pickle.load(open(VOCAB, 'rb'))
n_items = len(genres_list)
genre_to_idx = {g: i for i, g in enumerate(genres_list)}

net = Lorentz(n_items, DIM + 1)
net.load_state_dict(torch.load(CKPT, map_location='cpu'))
net.eval()

lorentz_table = net.table.weight.data.cpu().numpy()

def lorentz_distance(u, v):
    inner = -u[0]*v[0] + np.dot(u[1:], v[1:])
    inner = np.clip(-inner, 1 + 1e-6, None)
    return float(np.arccosh(inner))

def get_embedding(genre_name):
    if genre_name not in genre_to_idx:
        return None
    return lorentz_table[genre_to_idx[genre_name] + 1]

print(f'Loaded {n_items} genre embeddings')

## 2. Build genre popularity per market per year

For each (market, year) we sum the weekly streams of every chart song, attributed to each genre of that song's artist.

```
genre_popularity(genre, market, year) =
    sum of streams across all weeks
    for all songs whose artist has that genre
```

In [ ]:
artists_df = pd.read_csv(ARTISTS_CSV, sep='\t')

# parse genres from string representation of list
def parse_genres(s):
    try:
        return ast.literal_eval(s)
    except:
        return []

artists_df['genres_parsed'] = artists_df['genres'].apply(parse_genres)

# build name -> genres lookup
artist_genres = dict(zip(artists_df['name'].str.strip(),
                         artists_df['genres_parsed']))

print(f'Artists loaded: {len(artist_genres)}')
print('Sample:', list(artist_genres.items())[:3])

In [ ]:
from collections import defaultdict

popularity_rows = []
skipped_artists = set()

for market in MARKETS:
    for year in YEARS:
        charts_path = os.path.join(CHARTS_DIR, market, str(year))
        if not os.path.exists(charts_path):
            print(f'  Missing: {charts_path}')
            continue

        weekly_files = glob.glob(os.path.join(charts_path, '*.csv'))
        if not weekly_files:
            continue

        genre_streams = defaultdict(float)

        for fpath in weekly_files:
            try:
                chart = pd.read_csv(fpath, sep='\t')
            except Exception:
                continue

            for _, row in chart.iterrows():
                artist_name = str(row['artist']).strip()
                streams = row['streams']
                genres = artist_genres.get(artist_name, [])

                if not genres:
                    skipped_artists.add(artist_name)
                    continue

                for genre in genres:
                    genre_streams[genre] += streams

        for genre, total_streams in genre_streams.items():
            popularity_rows.append({
                'market':       market,
                'year':         year,
                'genre':        genre,
                'total_streams': total_streams
            })

    print(f'{market} done')

popularity_df = pd.DataFrame(popularity_rows)
popularity_df.to_csv(OUTPUT_POPULARITY, index=False)

print(f'\nGenre popularity table: {len(popularity_df)} rows')
print(f'Artists with no genres (skipped): {len(skipped_artists)}')
print()
print(popularity_df[popularity_df['market'] == 'us'][popularity_df['year'] == 2019]
      .sort_values('total_streams', ascending=False).head(10).to_string(index=False))

## 3. Build the enriched model dataset

For each (market, year) genre network file we:
- Remove self-loops
- Compute hyperbolic distance for each pair
- Attach `pop_source` and `pop_target` from the popularity table
- Add `market` and `year` columns

Then stack all markets and years into one CSV.

In [ ]:
# build a fast lookup: (market, year, genre) -> total_streams
pop_lookup = {}
for _, row in popularity_df.iterrows():
    pop_lookup[(row['market'], row['year'], row['genre'])] = row['total_streams']

all_rows = []
skipped_no_embedding = 0
skipped_no_popularity = 0

for market in MARKETS:
    for year in YEARS:
        net_path = os.path.join(GENRE_NET_DIR, market,
                                f'{market}-genre_network-{year}.csv')
        if not os.path.exists(net_path):
            print(f'  Missing genre network: {net_path}')
            continue

        gn = pd.read_csv(net_path, sep='\t')
        gn = gn[gn['source'] != gn['target']]  # remove self-loops

        for _, row in gn.iterrows():
            src, tgt = row['source'], row['target']

            # hyperbolic distance
            u = get_embedding(src)
            v = get_embedding(tgt)
            if u is None or v is None:
                skipped_no_embedding += 1
                continue

            dist = lorentz_distance(u, v)

            # genre popularity
            pop_src = pop_lookup.get((market, year, src))
            pop_tgt = pop_lookup.get((market, year, tgt))

            if pop_src is None or pop_tgt is None:
                skipped_no_popularity += 1
                continue

            all_rows.append({
                'market':       market,
                'year':         year,
                'source':       src,
                'target':       tgt,
                'weight':       row['weight'],
                'avg_streams':  row['avg_streams'],
                'distance':     dist,
                'pop_source':   pop_src,
                'pop_target':   pop_tgt,
                'log_streams':  np.log10(row['avg_streams'] + 1),
                'log_pop_src':  np.log10(pop_src + 1),
                'log_pop_tgt':  np.log10(pop_tgt + 1),
                'log_weight':   np.log10(row['weight'] + 1),
            })

model_df = pd.DataFrame(all_rows)
model_df.to_csv(OUTPUT_DATASET, index=False)

print(f'Model dataset: {len(model_df)} rows')
print(f'Skipped (no embedding): {skipped_no_embedding}')
print(f'Skipped (no popularity): {skipped_no_popularity}')
print()
print('Rows per market:')
print(model_df['market'].value_counts().to_string())
print()
print('Rows per year:')
print(model_df['year'].value_counts().sort_index().to_string())

## 4. Quick sanity check

In [ ]:
print('=== Dataset summary ===')
print(model_df[['avg_streams', 'distance', 'pop_source', 'pop_target', 'weight']]
      .describe().round(2))
print()
print('=== Sample rows ===')
print(model_df[['market', 'year', 'source', 'target',
                'distance', 'log_pop_src', 'log_pop_tgt',
                'log_weight', 'log_streams']]
      .head(10).to_string(index=False))
print()
print('=== Missing values ===')
print(model_df.isnull().sum())